In [23]:
import torch
import numpy as np
from PIL import Image
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import time

In [24]:
torch.manual_seed(0)

In [25]:
def polynomial_fun(w, x):
    """
    Evaluates a polynomial function given the weight vector w and an input scalar variable x.
    Args:
        w (torch.Tensor): Weight vector of size (M + 1, 1)
        x (torch.Tensor): Input scalar variables of size (N, 1)
    Returns:
        y (torch.Tensor): Output value of the polynomial function which has size (N, 1)
    """
    M = w.shape[0]
    powers = torch.arange(M, dtype=torch.float32)
    x_powers = torch.pow(x, powers)
    y = torch.matmul(x_powers, w)
    return y

In [26]:
def fit_polynomial_ls(x, t, M):
    """ 
    Implement a least squares solver for fitting polynomial functions using PyTorch's linear algebra modules.
    
    Args:
    - x (torch.Tensor): Input data points of shape (N, 1)
    - t (torch.Tensor): Target values of shape (N, 1)
    - M (int): Polynomial degree
    
    Returns:
    - w_hat (torch.Tensor): Optimum weight vector of shape (M+1, 1)
    """
    x_powers = torch.pow(x, torch.arange(M+1, dtype=torch.float32))
    w_hat = torch.linalg.lstsq(x_powers, t).solution
    return w_hat


In [55]:
def fit_polynomial_sgd(x, t, M, learning_rate, minibatch_size):
    """
    Fits a polynomial function using stochastic minibatch gradient descent.

    Args:
        x (torch.Tensor): Input data points of shape (N, 1)
        t (torch.Tensor): Target values of shape (N, 1)
        M (int): Polynomial degree
        learning_rate (float): Learning rate for gradient descent
        minibatch_size (int): Size of the minibatch

    Returns:
        w_opt (torch.Tensor): Optimum weight vector of shape (M+1, 1)
    """
    num_epochs = 2000
    x_powers = torch.pow(x, torch.arange(M+1, dtype=torch.float32))
    max_powers = (torch.max(torch.abs(x_powers), axis=0)).values
    #max_powers = x_powers[-1:]
    x_powers = x_powers/max_powers
    train_data = TensorDataset(x_powers, t)
    model = nn.Linear(M+1, 1, bias=False, dtype=torch.float32) 
    mse_loss = nn.MSELoss() 
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9) 
    #optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate) 
    epochs = []
    losses = []
    # Training loop
    for epoch in range(num_epochs):
        minibatch_data = DataLoader(train_data, batch_size=minibatch_size, shuffle=True)
        for x, y in minibatch_data:
            optimizer.zero_grad()  
            prediction = model(x)  
            loss = mse_loss(prediction, y)  
            loss.backward()  
            optimizer.step() 
        epochs.append(epoch)
        losses.append(loss.item())
        #Print loss every 100 epochs
        if (epoch + 1) % 100 == 0:
            print('Epoch: {} Loss {}'.format(epoch + 1, loss.item()))
    weight = model.weight
    w_opt = weight / max_powers
    return w_opt.reshape(M+1, 1)

In [56]:
# Define weight vector
w = torch.tensor([1, 2, 3], dtype=torch.float32).reshape(3, 1)

# Generate training set
x_train = 40.0 * (torch.rand(20, dtype=torch.float32) - 0.5).reshape(20, 1)
y_train = polynomial_fun(w, x_train)
noise_train = (0.5 * torch.randn(20, dtype=torch.float32)).reshape(20, 1)
t_train = y_train + noise_train

# Generate test set
x_test = 40.0 * (torch.rand(10, dtype=torch.float32) - 0.5).reshape(10, 1)
y_test = polynomial_fun(w, x_test)
noise_test = 0.5 * torch.randn(10, dtype=torch.float32).reshape(10, 1)
t_test = y_test + noise_test

# Compute optimum weight vector using fit_polynomial_ls for M=2,3,4 on the training set
time_ls_two = time.time()
w_hat_ls_two = fit_polynomial_ls(x_train, t_train, M=2)
time_ls_two = time.time() - time_ls_two
time_ls_three = time.time()
w_hat_ls_three = fit_polynomial_ls(x_train, t_train, M=3)
time_ls_three = time.time() - time_ls_three
time_ls_four = time.time()
w_hat_ls_four = fit_polynomial_ls(x_train, t_train, M=4)
time_ls_four = time.time() - time_ls_four

# Compute predicted target values for both training and test sets
# M = 2
y_hat_ls_train_two = polynomial_fun(w_hat_ls_two, x_train)
y_hat_ls_test_two = polynomial_fun(w_hat_ls_two, x_test)
# M = 3
y_hat_ls_train_three = polynomial_fun(w_hat_ls_three, x_train)
y_hat_ls_test_three = polynomial_fun(w_hat_ls_three, x_test)
# M = 4
y_hat_ls_train_four = polynomial_fun(w_hat_ls_four, x_train)
y_hat_ls_test_four = polynomial_fun(w_hat_ls_four, x_test)

# Difference between observed training data and the true polynomial curve
difference = t_train - y_train
mean_diff = torch.mean(difference)
std_diff = torch.std(difference)

print('Mean of difference (between observed training data and true polynomial curve) : {}'.format(mean_diff))
print('Standard Deviation of difference (between observed training data and true polynomial curve) : {}'.format(std_diff))

# Difference between LS-predicted values and the true polynomial curve
# M = 2
ls_difference_two = y_hat_ls_train_two - y_train
mean_diff_ls_two = torch.mean(ls_difference_two)
std_diff_ls_two = torch.std(ls_difference_two)
print('M=2: Mean of difference (between LS-predicted values and true polynomial curve) : {}'.format(mean_diff_ls_two))
print('M=2: Standard Deviation of difference (between LS-predicted values and true polynomial curve) : {}'.format(std_diff_ls_two))
# M = 3
ls_difference_three = y_hat_ls_train_three - y_train
mean_diff_ls_three = torch.mean(ls_difference_three)
std_diff_ls_three = torch.std(ls_difference_three)
print('M=3: Mean of difference (between LS-predicted values and true polynomial curve) : {}'.format(mean_diff_ls_three))
print('M=3: Standard Deviation of difference (between LS-predicted values and true polynomial curve) : {}'.format(std_diff_ls_three))
# M = 4
ls_difference_four = y_hat_ls_train_four - y_train
mean_diff_ls_four = torch.mean(ls_difference_four)
std_diff_ls_four = torch.std(ls_difference_four)
print('M=4: Mean of difference (between LS-predicted values and true polynomial curve) : {}'.format(mean_diff_ls_four))
print('M=4: Standard Deviation of difference (between LS-predicted values and true polynomial curve) : {}'.format(std_diff_ls_four))

Mean of difference (between observed training data and true polynomial curve) : -0.03502368927001953
Standard Deviation of difference (between observed training data and true polynomial curve) : 0.5231587886810303
M=2: Mean of difference (between LS-predicted values and true polynomial curve) : -0.03493504971265793
M=2: Standard Deviation of difference (between LS-predicted values and true polynomial curve) : 0.28036966919898987
M=3: Mean of difference (between LS-predicted values and true polynomial curve) : -0.03495064377784729
M=3: Standard Deviation of difference (between LS-predicted values and true polynomial curve) : 0.2803936004638672
M=4: Mean of difference (between LS-predicted values and true polynomial curve) : -0.034925542771816254
M=4: Standard Deviation of difference (between LS-predicted values and true polynomial curve) : 0.3336409330368042


In [59]:
# Compute optimum weight vector using fit_polynomial_sgd for M=2,3,4 on the training set
print("M=2:")
time_sgd_two = time.time()
w_hat_sgd_two = fit_polynomial_sgd(x_train, t_train, M=2, learning_rate=0.1, minibatch_size=10) 
time_sgd_two = time.time() - time_sgd_two
print("M=3:")
time_sgd_three = time.time()
w_hat_sgd_three = fit_polynomial_sgd(x_train, t_train, M=3, learning_rate=0.1, minibatch_size=10)
time_sgd_three = time.time() - time_sgd_three
print("M=4:")
time_sgd_four = time.time()
w_hat_sgd_four = fit_polynomial_sgd(x_train, t_train, M=4, learning_rate=0.1, minibatch_size=10)
time_sgd_four = time.time() - time_sgd_four

# Compute predicted target values for both training and test sets using fit_polynomial_sgd
# M = 2
y_hat_sgd_train_two = polynomial_fun(w_hat_sgd_two, x_train)
y_hat_sgd_test_two = polynomial_fun(w_hat_sgd_two, x_test)
# M = 3
y_hat_sgd_train_three = polynomial_fun(w_hat_sgd_three, x_train)
y_hat_sgd_test_three = polynomial_fun(w_hat_sgd_three, x_test)
# M = 4
y_hat_sgd_train_four = polynomial_fun(w_hat_sgd_four, x_train)
y_hat_sgd_test_four = polynomial_fun(w_hat_sgd_four, x_test)

# Difference between SGD-predicted values and the true polynomial curve
# M = 2
sgd_difference_two = y_hat_sgd_train_two - y_train
mean_diff_sgd_two = torch.mean(sgd_difference_two)
std_diff_sgd_two = torch.std(sgd_difference_two)
print('M=2: Mean of difference (between LS-predicted values and true polynomial curve) : {}'.format(mean_diff_sgd_two))
print('M=2: Standard Deviation of difference (between LS-predicted values and true polynomial curve) : {}'.format(std_diff_sgd_two))
# M = 3
sgd_difference_three = y_hat_sgd_train_three - y_train
mean_diff_sgd_three = torch.mean(sgd_difference_three)
std_diff_sgd_three = torch.std(sgd_difference_three)
print('M=3: Mean of difference (between LS-predicted values and true polynomial curve) : {}'.format(mean_diff_sgd_three))
print('M=3: Standard Deviation of difference (between LS-predicted values and true polynomial curve) : {}'.format(std_diff_sgd_three))
# M = 4
sgd_difference_four = y_hat_sgd_train_four - y_train
mean_diff_sgd_four = torch.mean(sgd_difference_four)
std_diff_sgd_four = torch.std(sgd_difference_four)
print('M=4: Mean of difference (between LS-predicted values and true polynomial curve) : {}'.format(mean_diff_sgd_four))
print('M=4: Standard Deviation of difference (between LS-predicted values and true polynomial curve) : {}'.format(std_diff_sgd_four))

M=2:
Epoch: 100 Loss 0.2466500699520111
Epoch: 200 Loss 0.1657457798719406
Epoch: 300 Loss 0.23209580779075623
Epoch: 400 Loss 0.18317899107933044
Epoch: 500 Loss 0.09435431659221649
Epoch: 600 Loss 0.1888481080532074
Epoch: 700 Loss 0.18880286812782288
Epoch: 800 Loss 0.24725449085235596
Epoch: 900 Loss 0.17881351709365845
Epoch: 1000 Loss 0.2321748286485672
Epoch: 1100 Loss 0.1754467487335205
Epoch: 1200 Loss 0.22189965844154358
Epoch: 1300 Loss 0.15263871848583221
Epoch: 1400 Loss 0.12500491738319397
Epoch: 1500 Loss 0.23082098364830017
Epoch: 1600 Loss 0.26694679260253906
Epoch: 1700 Loss 0.25200164318084717
Epoch: 1800 Loss 0.16799180209636688
Epoch: 1900 Loss 0.12270236015319824
Epoch: 2000 Loss 0.21364204585552216
M=3:
Epoch: 100 Loss 0.23738031089305878
Epoch: 200 Loss 0.2732677161693573
Epoch: 300 Loss 0.18189258873462677
Epoch: 400 Loss 0.24409174919128418
Epoch: 500 Loss 0.24666666984558105
Epoch: 600 Loss 0.21898405253887177
Epoch: 700 Loss 0.22235679626464844
Epoch: 800 Lo

In [61]:

# M = 2:
rmse_y_ls_two = compute_rmse(y_hat_ls_test_two, y_test)
rmse_y_sgd_two = compute_rmse(y_hat_sgd_test_two, y_test)
rmse_w_ls_two = compute_rmse(w_hat_ls_two, w)
rmse_w_sgd_two = compute_rmse(w_hat_sgd_two, w)


print("M=2: RMSE of y using LS is {}".format(rmse_y_ls_two))
print("M=2: RMSE of y using SGD is {}".format(rmse_y_sgd_two))
print("M=2: RMSE of w using LS is {}".format(rmse_w_ls_two))
print("M=2: RMSE of w using SGD is {}".format(rmse_w_sgd_two))


w_padded_three = nn.functional.pad(w, (0, 0, 0, 1), mode='constant', value=0)
# M = 3: 
rmse_y_ls_three = compute_rmse(y_hat_ls_test_three, y_test)
rmse_y_sgd_three = compute_rmse(y_hat_sgd_test_three, y_test)
rmse_w_ls_three = compute_rmse(w_hat_ls_three, w_padded_three)
rmse_w_sgd_three = compute_rmse(w_hat_sgd_three, w_padded_three)


print("M=3: RMSE of y using LS is {}".format(rmse_y_ls_three))
print("M=3: RMSE of y using SGD is {}".format(rmse_y_sgd_three))
print("M=3: RMSE of w using LS is {}".format(rmse_w_ls_three))
print("M=3: RMSE of w using SGD is {}".format(rmse_w_sgd_three))

w_padded_four = nn.functional.pad(w, (0, 0, 0, 2), mode='constant', value=0)
# M = 4: 
rmse_y_ls_four = compute_rmse(y_hat_ls_test_four, y_test)
rmse_y_sgd_four = compute_rmse(y_hat_sgd_test_four, y_test)
rmse_w_ls_four = compute_rmse(w_hat_ls_four, w_padded_four)
rmse_w_sgd_four = compute_rmse(w_hat_sgd_four, w_padded_four)


print("M=4: RMSE of y using LS is {}".format(rmse_y_ls_four))
print("M=4: RMSE of y using SGD is {}".format(rmse_y_sgd_four))
print("M=4: RMSE of w using LS is {}".format(rmse_w_ls_four))
print("M=4: RMSE of w using SGD is {}".format(rmse_w_sgd_four))

M=2: RMSE of y using LS is 0.22866961359977722
M=2: RMSE of y using SGD is 0.25458911061286926
M=2: RMSE of w using LS is 0.02543511800467968
M=2: RMSE of w using SGD is 0.04105113819241524
M=3: RMSE of y using LS is 0.2300247699022293
M=3: RMSE of y using SGD is 0.22730696201324463
M=3: RMSE of w using LS is 0.022098876535892487
M=3: RMSE of w using SGD is 0.03326338902115822
M=4: RMSE of y using LS is 0.31455111503601074
M=4: RMSE of y using SGD is 0.2707284688949585
M=4: RMSE of w using LS is 0.09087889641523361
M=4: RMSE of w using SGD is 0.12407663464546204


In [63]:
print("M = 2: Time spent training ls: {}".format(time_ls_two))
print("M = 2: Time spent training sgd: {}".format(time_sgd))
print("M = 3: Time spent training ls: {}".format(time_ls_three))
print("M = 3: Time spent training sgd: {}".format(time_sgd))
print("M = 4: Time spent training ls: {}".format(time_ls_four))
print("M = 4: Time spent training sgd: {}".format(time_sgd))

M = 2: Time spent training ls: 0.00046443939208984375
M = 2: Time spent training sgd: 0.3822004795074463
M = 3: Time spent training ls: 0.00026297569274902344
M = 3: Time spent training sgd: 0.3822004795074463
M = 4: Time spent training ls: 0.0002644062042236328
M = 4: Time spent training sgd: 0.3822004795074463


In [64]:

def fit_polynomial_sgd_weight_regularised(x, t, M, lr, batch_size):
    """ 
    Implementation of stochastic gradient descent (SGD) for polynomial function fitting with an added penalty 
    term to the loss dependent on the L2-norm of the weight vector to encourage small weights.

    Args:
        x (torch.Tensor): Vectorized scalars in x (shape: number of data points by 1).
        t (torch.Tensor): Target values t (shape: number of data points by 1).
        M (int): Maximum degree of the class of polynomials being optimized over.
        lr (float): Learning rate.
        batch_size (int): Batch size.

    Returns:
        torch.Tensor: Optimum weight vector 𝐰̂ (shape: 1 by M+1) found by applying SGD to minimize 
                      squared distance of t and y=𝐰̂x while also trying to minimize model complexity 
                      (in the form of polynomial degree).
    """
    # Print message indicating the start of training
    print(". \n" * 5)
    print("-" * 20 + "Training Start" + "-" * 20)
    print("Starting SGD training to determine the optimal weight vector, also optimizing over polynomial degree.")
    print("Training for 150,000 epochs.")
    print("Maximum polynomial degree considered: " + str(M))

    
    # Scaling inputs to prevent exploding gradients
    powers_of_x = torch.pow(x, torch.arange(M+1, dtype=torch.float32))
    maximum_powers_of_x = torch.max(torch.abs(powers_of_x), axis=0).values
    powers_of_x = torch.div(powers_of_x, maximum_powers_of_x)
    data_set = TensorDataset(powers_of_x, t)
    
    # Define model, loss function, and optimizer
    model = nn.Linear(M+1, 1, bias=False, dtype=torch.float32)
    loss_func = nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    
    alpha = 10**(-M)  # Weight regularization alpha
    
    # Lists to store epoch and loss for plotting
    epoch_list = []
    loss_list = []

    # Training loop
    for epoch in range(150_000):
        # Data batches
        data_batches = DataLoader(data_set, batch_size=batch_size, shuffle=True)
        for input, ground_truth in data_batches:
            optimizer.zero_grad()  # Clear gradients
            prediction = model(input)  # Forward pass

            # Regularization term using L2-norm
            regularisation_term = torch.sum(torch.square(torch.div(model.weight, maximum_powers_of_x)))
            # Loss function with weight regularization
            loss = loss_func(prediction, ground_truth) + alpha * regularisation_term
            loss.backward()  # Backpropagation
            optimizer.step()  # Update weights

        # Print loss every 10,000 epochs
        if epoch % 10_000 == 0:
            # Set weights that are not influential to 0 every 10,000 epochs
            flag = torch.abs(torch.div(model.weight, maximum_powers_of_x)) >= 1e-3
            model.weight.data = model.weight * flag
            print("Epoch: " + str(epoch) + ", MSE + L2 weight regularization loss: " + str(loss.item()))

        # Append epoch and loss for plotting
        epoch_list.append(epoch)
        loss_list.append(loss.item())

    print("Epoch: " + str(epoch) + ", MSE + L2 weight regularization loss: " + str(loss.item()))
    
    # Set weights to 0 for those not influential
    flag = torch.abs(torch.div(model.weight, maximum_powers_of_x)) >= 1e-3
    model.weight.data = model.weight * flag
    w_hat = model.weight
    
    # Rescale the weight
    print("-" * 20 + "end" + "-" * 20)
    w_hat = torch.div(w_hat, maximum_powers_of_x).T  # Transpose to match polynomial_fun(w, x) convention
    

    return w_hat




In [65]:
def main():
    
    #Use polynomial_fun (𝑀 =10, 𝐰=[1,2,3,4,5]T) to generate a training set and a test set, in the 
    #form of respectively sampled 100 and 50 pairs of 𝑥,𝑥𝜖[−20,20], and 𝑡. The observed 𝑡 values 
    #are obtained by adding Gaussian noise (standard deviation being 0.2) to 𝑦.
    
    temp1 = torch.arange(3, dtype=torch.float32)
    temp2 = torch.tensor(1, dtype=torch.float32)
    w = torch.add(temp1, temp2)
    w = w.reshape(w.shape[0], -1)
    del temp1, temp2
    w = torch.tensor([1,2,3,4,5], dtype=torch.float32).reshape(5,1)

    

    #training set
    x_train = 40.0*(torch.rand(100, dtype=torch.float32) - 0.5).reshape(100,1)
    y_train = polynomial_fun(w,x_train)
    noise_train = (0.2*torch.randn(100, dtype=torch.float32)).reshape(100,1)
    t_train = y_train+noise_train
    del noise_train


    #testing set
    x_test = 40.0*(torch.rand(50, dtype=torch.float32) - 0.5).reshape(50,1)
    y_test = polynomial_fun(w,x_test)
    noise_test = 0.2*torch.randn(50, dtype=torch.float32).reshape(50,1)
    t_test = y_test + noise_test
    del noise_test

    # Report the optimized 𝑀 value and the mean (and standard deviation) in difference between the model-predicted values and the underlying “true” polynomial curve
    M_max = 10 # Maximum polynomial degree allowed during fitting
    w_hat_sgd = fit_polynomial_sgd_weight_regularised(x_train, t_train, M_max, 0.5, 25)  # Batch size=25, learning rate = 0.5

    # Print optimized weight vector and optimal polynomial degree
    print("-" * 40 + "Optimized Results" + "-" * 40)
    print("Optimized weight vector:")
    print(w_hat_sgd.tolist())
    # Find the optimal polynomial degree after training
    optimal_M = torch.max(torch.nonzero(w_hat_sgd)[:,0])
    print("\nOptimized degree of the polynomial:", optimal_M.item()+1)
    print("-" * 87)

    # Compute predicted values for both training and testing sets
    y_hat_sgd_train = polynomial_fun(w_hat_sgd, x_train)
    y_hat_sgd_test = polynomial_fun(w_hat_sgd, x_test)

    # Compute difference between predicted values and true polynomial for training set
    difference = y_hat_sgd_train - y_train
    std_difference, mean_difference = torch.std_mean(difference)
    print(". \n" * 5)
    print("-" * 20 + "Difference between predicted values (on training set) and true polynomial" + "-" * 20)
    print("Mean difference: ", mean_difference.tolist())
    print("Standard deviation: ", std_difference.tolist())
    print("-" * 20 + "end" + "-" * 20)
    del difference, std_difference, mean_difference

    # Compute difference between predicted values and true polynomial for testing set
    difference = y_hat_sgd_test - y_test
    std_difference, mean_difference = torch.std_mean(difference)
    print(". \n" * 5)
    print("-" * 20 + "Difference between predicted values (on testing set) and true polynomial" + "-" * 20)
    print("Mean difference: ", mean_difference.tolist())
    print("Standard deviation: ", std_difference.tolist())
    print("-" * 20 + "end" + "-" * 20)
    del difference, std_difference, mean_difference



if __name__=="__main__":
    main()


. 
. 
. 
. 
. 

--------------------Training Start--------------------
Starting SGD training to determine the optimal weight vector, also optimizing over polynomial degree.
Training for 150,000 epochs.
Maximum polynomial degree considered: 10
Epoch: 0, MSE + L2 weight regularization loss: 3780332544.0
Epoch: 10000, MSE + L2 weight regularization loss: 987032.375
Epoch: 20000, MSE + L2 weight regularization loss: 325714.625
Epoch: 30000, MSE + L2 weight regularization loss: 99094.15625
Epoch: 40000, MSE + L2 weight regularization loss: 40566.7578125
Epoch: 50000, MSE + L2 weight regularization loss: 22440.83203125
Epoch: 60000, MSE + L2 weight regularization loss: 1515.564697265625
Epoch: 70000, MSE + L2 weight regularization loss: 81.49061584472656
Epoch: 80000, MSE + L2 weight regularization loss: 3.378777503967285
Epoch: 90000, MSE + L2 weight regularization loss: 0.2494441270828247
Epoch: 100000, MSE + L2 weight regularization loss: 0.07246523350477219
Epoch: 110000, MSE + L2 weight